# Examples of Tracking and Curtain Visualization

This notebook utilizes concatenated data from the Pacific hake survey to create echograms and corresponding track and draped curtain visualizations. The hake survey data has been integrated with geographical coordinates for comprehensive analysis.

Interactions between echograms and tracks are configured explicitly in the notebook using Panel and HoloViews streams rather than being stored on the xarray accessor.

## The Significance of Tracking and Curtain Plotting

- **Echogram-Controlled Selection**: Fisheries scientists can select a section of an echogram and visualize where the corresponding echosounder data was collected on the map. This helps them assess the fish's association with bathymetry or other environmental variables.

- **Track Visualization**: Fisheries scientists can visualize the ship track on a map and relate geographic position to the corresponding acoustic observations.

- **Curtain Visualization**: Fisheries scientists can choose to represent longer sections of echograms or ship tracks on the map as curtains. This feature offers a more comprehensive view of how fish aggregations vary spatially.

- **Region of Interest Selection**: Fisheries scientists can select a specific region of the echogram and examine the corresponding `Sv` (volume backscattering strength) data in more detail.

- **Exporting MVBS Dataset**: The selected `MVBS` (mean volume backscattering strength) dataset can be saved to a separate file for further analysis or sharing.

## Import Packages and Data 

In [1]:
import holoviews as hv
import panel as pn
import xarray as xr

from echoshader.app import get_box_plot, get_box_stream

pn.extension("bokeh", comms="default")

In [2]:
from urllib import request

# Calibratd data is stored in Google Drive
url = 'https://drive.google.com/uc?export=download&id=197D0MW-bHaF6mZLcQwyr4zqyEHIfwsep'

def urllib_download():
    request.urlretrieve(url, 'concatenated_MVBS.nc')

urllib_download() 

# Load sample data for testing
MVBS_ds = xr.open_mfdataset(
    paths="concatenated_MVBS.nc",
    data_vars="minimal",
    coords="minimal",
    combine="by_coords",
)

MVBS_ds

<xarray.Dataset> Size: 4MB
Dimensions:            (channel: 4, ping_time: 875, echo_range: 150)
Coordinates:
  * channel            (channel) <U37 592B 'GPT  18 kHz 009072058c8d 1-1 ES18...
  * ping_time          (ping_time) datetime64[ns] 7kB 2017-07-24T19:30:00 ......
    time1              (ping_time) datetime64[ns] 7kB dask.array<chunksize=(875,), meta=np.ndarray>
  * echo_range         (echo_range) float64 1kB 0.0 5.0 10.0 ... 740.0 745.0
Data variables:
    Sv                 (channel, ping_time, echo_range) float64 4MB dask.array<chunksize=(4, 875, 150), meta=np.ndarray>
    frequency_nominal  (channel) float64 32B dask.array<chunksize=(4,), meta=np.ndarray>
    longitude          (ping_time) float64 7kB dask.array<chunksize=(875,), meta=np.ndarray>
    latitude           (ping_time) float64 7kB dask.array<chunksize=(875,), meta=np.ndarray>
Attributes:
    processing_software_name:     echopype
    processing_software_version:  0.7.1
    processing_time:              2023-05-30T17:40:45Z
    processing_function:          commongrid.compute_MVBS

## Track Demonstration

Users have the flexibility to customize the map tile used for the track plot. Different tile options can be explored in the [HoloViews Tiles gallery](https://holoviews.org/reference/elements/bokeh/Tiles.html).

Here's a breakdown of the visual elements:
- A blue point signifies the starting point of the track.
- A red line depicts the course of the ship.

It's worth noting that when echo data originates from a moored platform, only one blue point will be displayed.

In [3]:
track = MVBS_ds.eshader.track(
    tile="OpenTopoMap",
)

eg = MVBS_ds.eshader.echogram(
    channel="GPT  18 kHz 009072058c8d 1-1 ES18-11",
).opts(
    cmap="jet",
    width=1250,
    height=450,
)

layout = pn.Column(track, eg)
layout

Column
    [0] HoloViews(Overlay, height=300, sizing_mode='fixed', width=300)
    [1] HoloViews(Image, height=450, sizing_mode='fixed', width=1250)

The map tile can also be controlled with an explicit Panel widget. The widget state now lives in the notebook rather than on the xarray accessor.

In [4]:
tile_select = pn.widgets.Select(
    name="Tile",
    options=["OpenTopoMap", "OSM"],
    value="OpenTopoMap",
)

def track_view(tile):
    return MVBS_ds.eshader.track(
        tile=tile,
    )

track_bound = pn.bind(
    track_view,
    tile=tile_select,
)

track_panel = pn.Row(
    tile_select,
    track_bound,
)

track_panel

Row
    [0] Select(label='Tile', name='Tile', options=['OpenTopoMap', 'OSM'], value='OpenTopoMap')
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

## Track Demonstration with Echogram-Controlled Selection

Previously, this behavior was selected through accessor-owned `control` state. In the refactored architecture, the echogram selection stream and the link to the track are defined explicitly in the notebook.

Use the `Box Select` tool on the echogram. The track is then generated from the corresponding selected ping-time range.

In [5]:
def data_from_box(ds, bounds, vert_dim="echo_range"):
    if bounds is None:
        return ds

    left, bottom, right, top = bounds

    return ds.sel(
        ping_time=slice(left, right),
        **{vert_dim: slice(min(bottom, top), max(bottom, top))},
    )


eg_with_echogram_mode = MVBS_ds.eshader.echogram(
    channel="GPT  38 kHz 009072058146 2-1 ES38B",
).opts(
    cmap="jet",
    colorbar=True,
    tools=["box_select", "hover"],
    width=1250,
    height=450,
)

box_stream = get_box_stream(eg_with_echogram_mode)
box_overlay = get_box_plot(box_stream)


@pn.depends(box_stream.param.bounds)
def track_from_echogram(bounds):
    selected_ds = data_from_box(MVBS_ds, bounds)
    return selected_ds.eshader.track(
        tile="OpenTopoMap",
    )


pn.Column(
    eg_with_echogram_mode * box_overlay,
    track_from_echogram,
)

Column
    [0] HoloViews(DynamicMap, height=450, sizing_mode='fixed', width=1250)
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

## Box Selection

The selected echogram region can be converted directly into the corresponding xarray dataset. Selection state is now read from the HoloViews stream instead of from the xarray accessor.

Use the plot's reset tool to clear the visual selection.

In [6]:
data_from_box_select = data_from_box(
    MVBS_ds,
    box_stream.bounds,
)

data_from_box_select

<xarray.Dataset> Size: 4MB
Dimensions:            (channel: 4, ping_time: 875, echo_range: 150)
Coordinates:
  * channel            (channel) <U37 592B 'GPT  18 kHz 009072058c8d 1-1 ES18...
  * ping_time          (ping_time) datetime64[ns] 7kB 2017-07-24T19:30:00 ......
    time1              (ping_time) datetime64[ns] 7kB dask.array<chunksize=(875,), meta=np.ndarray>
  * echo_range         (echo_range) float64 1kB 0.0 5.0 10.0 ... 740.0 745.0
Data variables:
    Sv                 (channel, ping_time, echo_range) float64 4MB dask.array<chunksize=(4, 875, 150), meta=np.ndarray>
    frequency_nominal  (channel) float64 32B dask.array<chunksize=(4,), meta=np.ndarray>
    longitude          (ping_time) float64 7kB dask.array<chunksize=(875,), meta=np.ndarray>
    latitude           (ping_time) float64 7kB dask.array<chunksize=(875,), meta=np.ndarray>
Attributes:
    processing_software_name:     echopype
    processing_software_version:  0.7.1
    processing_time:              2023-05-30T17:40:45Z
    processing_function:          commongrid.compute_MVBS

## Curtain Visualization

For sonar data collected from mobile platforms like ships, imagine the echogram as a curtain draped along the GPS track of the vessel.

Users can customize the curtain with the following options:

- `channel`: Select the frequency channel to plot.
- `ratio`: Adjust the curtain ratio for z-axis spacing.
- `cmap`: Select the colormap.
- `clim`: Set the Sv (volume backscattering strength) range.

In the refactored architecture, these controls are explicit Panel widgets created in the notebook. They are not stored on the xarray accessor.

If you don't see the panel displayed when using the PyVista engine, make sure to set up the `pyvista` extension for Panel:

```python
pn.extension("pyvista")
```

Additionally, when working on a Linux/EC2 instance, PyVista may require a virtual framebuffer. See the [PyVista reference](https://github.com/pyvista/pyvista/issues/177).

```python
pv.start_xvfb()
```

You may also need the relevant Linux graphics libraries:

```text
sudo apt install -y libgl1-mesa-glx xvfb libxrender-dev
```

In [7]:
import sys

import pyvista as pv

pn.extension("pyvista")

if sys.platform.startswith("linux"):
    pv.start_xvfb()

colormap = pn.widgets.LiteralInput(
    name="Colormap",
    value="jet",
)

Sv_range_slider = pn.widgets.EditableRangeSlider(
    name="Sv Range Slider",
    start=-120,
    end=0,
    value=(-80, -30),
)

channel_select = pn.widgets.Select(
    name="Channel",
    options=list(MVBS_ds.channel.values),
    value=MVBS_ds.channel.values[0],
)

curtain_ratio = pn.widgets.FloatInput(
    name="Curtain Ratio",
    value=0.001,
    step=0.001,
)


def curtain_view(channel, ratio, cmap, clim):
    return MVBS_ds.eshader.curtain(
        channel=channel,
        ratio=ratio,
        engine="pyvista",
        cmap=cmap,
        clim=clim,
    )


curtain_bound = pn.bind(
    curtain_view,
    channel=channel_select,
    ratio=curtain_ratio,
    cmap=colormap,
    clim=Sv_range_slider,
)

curtain_panel = pn.Row(
    pn.Column(
        colormap,
        Sv_range_slider,
        channel_select,
        curtain_ratio,
    ),
    curtain_bound,
)

curtain_panel

BokehModel(combine_events=True, render_bundle={'docs_json': {'0b5f556a-cadd-4792-ac28-4b584e6c494a': {'version…

Here is an example picture.

## Applying Plot Customizations

HoloViews visualizations such as echograms and tracks can be customized using native HoloViews `.opts()` settings.

For detailed information about `HoloViews options`, please refer to this [link](https://holoviews.org/user_guide/Applying_Customizations.html#option-list-syntax).

Curtain visualizations return native Plotly/PyVista objects, so curtain-specific display customization follows the corresponding backend rather than the HoloViews `.opts()` interface.